In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

In [ ]:
from sklearn.datasets import load_digits
X, y = load_digits(return_X_y=True)


In [ ]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (1797, 64)
Shape of y: (1797,)


In [ ]:
print(X)

[[ 0.  0.  5. ...  0.  0.  0.]
 [ 0.  0.  0. ... 10.  0.  0.]
 [ 0.  0.  0. ... 16.  9.  0.]
 ...
 [ 0.  0.  1. ...  6.  0.  0.]
 [ 0.  0.  2. ... 12.  0.  0.]
 [ 0.  0. 10. ... 12.  1.  0.]]


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
scalaler = StandardScaler()
X_train_scaled = scalaler.fit_transform(X_train)
X_test_scaled = scalaler.transform(X_test)

In [7]:

lda = LinearDiscriminantAnalysis(n_components=9)


using sklearn.discriminant_anlalysis 


In [33]:
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

y_pred = lda.predict(X_test)
print("Accuracy (LDA from scratch):", metrics.accuracy_score(y_test, y_pred))

Accuracy (LDA from scratch): 0.9527777777777777


LDA from scratch

In [34]:
import numpy as np
classes = np.unique(y_train)
n_features = X_train.shape[1]
n_samples = X_train.shape[0]

In [35]:
means = {}
priors = {}
cov = np.zeros((n_features, n_features))

for c in classes:
    X_c = X_train[y_train == c]
    means[c] = np.mean(X_c, axis=0)
    priors[c] = X_c.shape[0] / n_samples
    cov += np.cov(X_c, rowvar=False) * (X_c.shape[0] - 1)

# Shared covariance
cov /= (n_samples - len(classes))
cov_inv = np.linalg.pinv(cov)


In [36]:
def discriminant(x, mean, cov_inv, prior):
    return (
        x @ cov_inv @ mean
        - 0.5 * mean @ cov_inv @ mean
        + np.log(prior)
    )


In [37]:
y_pred = []

for x in X_test:
    scores = [
        discriminant(x, means[c], cov_inv, priors[c])
        for c in classes
    ]
    y_pred.append(classes[np.argmax(scores)])


In [39]:
print("Accuracy (sklearn-equivalent scratch LDA):",
      metrics.accuracy_score(y_test, y_pred))


Accuracy (sklearn-equivalent scratch LDA): 0.9527777777777777
